# P4-5 · RAG nâng cao

Toàn bộ code của bài, chạy từ trên xuống như trên trang bài học. Kết quả đã chạy sẵn được giữ trong notebook; bấm **Runtime → Run all** để chạy lại. Ô có dòng đầu `# Cần GPU` thì đổi **Runtime → Change runtime type → T4 GPU**; ô cần API key thì đặt khóa trong Colab Secrets.

In [ ]:
!pip install -q sentence-transformers

## Lý thuyết

In [ ]:
def rrf(lists, k=60):
    score = {}
    for lst in lists:
        for rank, doc in enumerate(lst, start=1):
            score[doc] = score.get(doc, 0) + 1 / (k + rank)
    return sorted(score, key=score.get, reverse=True)

# ví dụ nhỏ: hai bảng xếp hạng của cùng một câu hỏi
bm25_top50 = ["đổi trả 7 ngày", "phí ship", "bảo hành 12 tháng", "đổi size"]
vector_top50 = ["đổi size", "đổi trả 7 ngày", "hoàn tiền", "phí ship"]
candidates = rrf([bm25_top50, vector_top50])[:50]
candidates          # tài liệu đứng cao ở cả hai danh sách được đẩy lên đầu

['đổi trả 7 ngày', 'đổi size', 'phí ship', 'bảo hành 12 tháng', 'hoàn tiền']


In [ ]:
# Cần mạng: tải reranker từ Hugging Face, chạy trên Colab
from sentence_transformers import CrossEncoder
reranker = CrossEncoder("BAAI/bge-reranker-v2-m3")
pairs = [(question, chunks[i]["text"]) for i in candidates]         # 50 cặp
scores = reranker.predict(pairs, batch_size=16)
top5 = [candidates[i] for i in scores.argsort()[::-1][:5]]

## Bài tập 5: Pipeline hai tầng {code}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
# Cần mạng và index ở bài P4.2: chạy trên Colab
import numpy as np
from sentence_transformers import CrossEncoder
reranker = CrossEncoder("BAAI/bge-reranker-v2-m3")

def rrf(lists, k=60):
    score = {}
    for lst in lists:
        for rank, d in enumerate(lst, 1):
            score[d] = score.get(d, 0) + 1 / (k + rank)
    return sorted(score, key=score.get, reverse=True)

def retrieve(question, k_final=5, k_wide=50):
    qv = emb.encode([question], normalize_embeddings=True).astype("float32")
    vec = list(index.search(qv, k_wide)[1][0])
    kw = list(np.argsort(-bm25.get_scores(tokenize(question)))[:k_wide])
    cand = rrf([vec, kw])[:k_wide]
    scores = reranker.predict([(question, chunks[i]["text"]) for i in cand])
    return [cand[i] for i in np.argsort(-scores)[:k_final]]

## Bài tập 6: Viết lại câu hỏi để tìm được nhiều hơn {ứng dụng}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import math
docs = ["Thời hạn đổi trả là 7 ngày kể từ khi nhận hàng", "Phí vận chuyển nội thành 30.000 đồng",
        "Hoàn tiền qua tài khoản ngân hàng trong 5 ngày làm việc", "Bảo hành khung bàn 12 tháng",
        "Đơn vị vận chuyển giao hàng từ 8 giờ đến 20 giờ", "Hủy đơn miễn phí trước khi đơn được bàn giao vận chuyển"]
hoi = [("bao lâu thì được trả tiền lại", 2), ("ship nội thành bao nhiêu tiền", 1), ("mấy giờ shipper tới", 4),
       ("mua rồi có được trả lại không", 0), ("không muốn mua nữa thì bỏ đơn được không", 5)]
DONG_NGHIA = {"trả tiền lại": "hoàn tiền", "ship": "phí vận chuyển", "shipper": "đơn vị vận chuyển giao hàng",
              "trả lại": "đổi trả", "bỏ đơn": "hủy đơn", "mấy giờ": "từ giờ đến giờ"}
toks = [d.lower().split() for d in docs]
N = len(docs)
idf = {w: math.log(1 + N / sum(w in t for t in toks)) for t in toks for w in t}

def tim(q, k=3):
    diem = [sum(idf.get(w, 0) for w in set(q.lower().split()) if w in t) for t in toks]
    return sorted(range(N), key=lambda i: -diem[i])[:k]

def bien_the(q):
    # TODO: trả về [câu gốc, câu thay cụm đồng nghĩa, câu gốc + cụm chuyên môn]
    return [q]

In [ ]:
def bien_the(q):
    thay, them = q, []
    for cum, ct in DONG_NGHIA.items():
        if cum in q:
            thay = thay.replace(cum, ct); them.append(ct)
    return [q, thay, q + " " + " ".join(them)]

def tim_mq(q, k=3):
    diem = {}
    for v in bien_the(q):
        for h, i in enumerate(tim(v, k)):
            diem[i] = diem.get(i, 0) + 1 / (60 + h + 1)
    return sorted(diem, key=lambda i: -diem[i])[:k]

recall_goc = sum(d in tim(q, 1) for q, d in hoi) / len(hoi)
recall_mq = sum(d in tim_mq(q, 1) for q, d in hoi) / len(hoi)
for q, d in hoi:
    print(f"{q:42s} gốc {tim(q)} | multi-query {tim_mq(q)} | đúng {d}")
print(f"recall@1: gốc {recall_goc:.2f} → multi-query {recall_mq:.2f}")

bao lâu thì được trả tiền lại              gốc [0, 2, 5] | multi-query [2, 0, 5] | đúng 2
ship nội thành bao nhiêu tiền              gốc [1, 2, 0] | multi-query [1, 5, 4] | đúng 1
mấy giờ shipper tới                        gốc [4, 0, 1] | multi-query [4, 1, 5] | đúng 4
mua rồi có được trả lại không              gốc [0, 5, 1] | multi-query [0, 5, 1] | đúng 0
không muốn mua nữa thì bỏ đơn được không   gốc [5, 4, 0] | multi-query [5, 4, 0] | đúng 5
recall@1: gốc 0.80 → multi-query 1.00


## Bài tập 7: Chunking theo cấu trúc tài liệu {ứng dụng}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
md = "\n".join([
    "# Chính sách",
    "## Đổi trả", "### Thời hạn", "Trong 7 ngày kể từ khi nhận hàng.",
    "### Điều kiện", "Sản phẩm còn nguyên tem, chưa qua sử dụng.",
    "## Bảo hành", "### Thời hạn", "12 tháng cho khung, 6 tháng cho phụ kiện.",
    "### Nơi nhận", "Tại cửa hàng hoặc gửi qua bưu điện.",
])

def chunk_markdown(md):
    # TODO: trả về list {"duong_dan": "Bảo hành > Thời hạn", "text": "Bảo hành > Thời hạn: 12 tháng ..."}
    return []

def tim(q, chunks):
    tu = set(q.lower().split())
    return max(chunks, key=lambda c: len(tu & set(c["text"].lower().replace(">", " ").replace(":", " ").split())))

In [ ]:
def chunk_markdown(md):
    duong, out = [], []
    for line in md.splitlines():
        if line.startswith("### "):
            duong = duong[:1] + [line[4:].strip()]
        elif line.startswith("## "):
            duong = [line[3:].strip()]
        elif line.strip() and not line.startswith("#"):
            dd = " > ".join(duong)
            out.append({"duong_dan": dd, "text": f"{dd}: {line.strip()}"})
    return out

chunks = chunk_markdown(md)
tho = [{"duong_dan": "", "text": l} for l in md.splitlines() if l.strip() and not l.startswith("#")]
for c in chunks:
    print(c["text"])
print("cấu trúc →", tim("thời hạn bảo hành bao lâu", chunks)["text"])
print("cắt thô  →", tim("thời hạn bảo hành bao lâu", tho)["text"])

Đổi trả > Thời hạn: Trong 7 ngày kể từ khi nhận hàng.
Đổi trả > Điều kiện: Sản phẩm còn nguyên tem, chưa qua sử dụng.
Bảo hành > Thời hạn: 12 tháng cho khung, 6 tháng cho phụ kiện.
Bảo hành > Nơi nhận: Tại cửa hàng hoặc gửi qua bưu điện.
cấu trúc → Bảo hành > Thời hạn: 12 tháng cho khung, 6 tháng cho phụ kiện.
cắt thô  → Trong 7 ngày kể từ khi nhận hàng.


## Bài tập 8: Hai tầng: lọc rộng rồi xếp lại kỹ {ứng dụng}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import numpy as np
rng = np.random.default_rng(0)
Q, D, MS_MOI_DOC = 50, 200, 15
lien_quan = rng.integers(0, D, Q)                   # chỉ số tài liệu đúng của mỗi truy vấn
that = np.zeros((Q, D)); that[np.arange(Q), lien_quan] = 1
tang1 = that * 1.0 + rng.normal(0, 0.6, (Q, D))     # nhanh, nhiễu nhiều
tang2 = that * 1.0 + rng.normal(0, 0.15, (Q, D))    # chậm, ít nhiễu
# TODO: hit3_tang1 (tỉ lệ truy vấn có tài liệu đúng trong top-3 tầng 1), hit3_hai_tang, do_tre

In [ ]:
def hit3(q, hai_tang):
    ung_vien = np.argsort(-tang1[q])[:20 if hai_tang else 3]
    if hai_tang:
        ung_vien = ung_vien[np.argsort(-tang2[q, ung_vien])][:3]
    return lien_quan[q] in ung_vien

hit3_tang1 = np.mean([hit3(q, False) for q in range(Q)])
hit3_hai_tang = np.mean([hit3(q, True) for q in range(Q)])
trong_top20 = np.mean([lien_quan[q] in np.argsort(-tang1[q])[:20] for q in range(Q)])
do_tre = 20 * MS_MOI_DOC
print(f"chỉ tầng 1: {hit3_tang1:.2f} | hai tầng: {hit3_hai_tang:.2f} (trần: {trong_top20:.2f} có trong top-20) | +{do_tre} ms")

chỉ tầng 1: 0.28 | hai tầng: 0.56 (trần: 0.56 có trong top-20) | +300 ms
